In [1]:
import numpy as np
import pandas as pd
import scipy
from sklearn.linear_model import LinearRegression

This lab uses data describing FIFA soccer players. The comes from a Kaggle challenge (<a href="https://www.kaggle.com/datasets/moradi/fifa-stats">FIFA Data Set </a>) that includes things like players' wages, popularity, and information about their skills and abilities.

In [2]:
fifa = pd.read_csv(
    "./fifa/fifa_players.csv",
    usecols=[
        "player_name","club_name",
        "Overall rating",
        "Height","Weight",
        "Wage","Total attacking",
        "Crossing",'Short passing',"Volleys","Dribbling",'Total defending',
        "Growth",'International reputation',
    ],
    dtype={
        "Overall rating":object,
        "Total attacking":object,
        "Crossing":object,
        'Short passing':object,
        "Volleys":object,
        "Dribbling":object,
        'Total defending':object,
    }
)

def clean(s): return s.split("+")[0].split('-')[0] if type(s) == str else s 
for f in ["Overall rating","Crossing",'Short passing',"Volleys","Dribbling",'Total defending',
          "Total attacking","Growth"]:
    fifa[f] = pd.to_numeric(fifa[f].apply(clean))
fifa["Wage"] = pd.to_numeric(fifa["Wage"].apply(lambda s: s[1:].replace("K","000")))
fifa.head()

,player_name,Overall rating,Height,Weight,Growth,Wage,Total attacking,Crossing,Short passing,Volleys,Dribbling,Total defending,International reputation,club_name
0,Erling Haaland,91,195,94,3,340000,393,47,77,90,79,114,5,Manchester City
1,Kylian Mbappe,91,182,75,3,230000,415,78,86,84,93,92,5,Paris Saint Germain
2,Kevin De Bruyne,91,181,75,0,350000,412,95,94,83,86,189,5,Manchester City
3,Rodrigo Hernandez Cascante,90,191,82,1,260000,373,71,92,59,81,257,4,Manchester City
4,Harry Kane,90,188,85,0,170000,440,80,87,89,82,130,5,nchen


**Question 1:** Out of the numeric variables in the fifa dataframe (i.e., excluding club_name and player_name), which variables has the largest OLS regression coefficient when predicting players' wages?

Write code using LinearRegression from Sklearn to calculate the regression coefficients. Return a Pandas Series with the regression coefficients as the data and the variable names as the index. Since the variables have different scales (e.g., Overall rating is on a 100 point scale while players' heights are measured in centimeters), standardize and center both input and output variables before fitting the model.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

def getOlsCoefs():
    '''
    Standardize and use OLS model to predict the target variable using the 
    independent variables from the `inputs` list.
    '''
    # All numeric predictor columns (exclude target and non-numeric)
    inputs = list(fifa.drop(columns=["Wage", "club_name", "player_name"]).columns)
    
    # Drop rows with any NaN values
    X = fifa[inputs].dropna()
    y = fifa.loc[X.index, "Wage"]
    
    # Standardize (center + scale) both X and y
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1)).ravel()
    
    # Fit OLS model
    ols = LinearRegression()
    ols.fit(X_scaled, y_scaled)
    
    # Return coefficients as a named Pandas Series
    coefs = pd.Series(
        data=ols.coef_,
        index=inputs,
        name='ols'
    )
    
    return coefs

# Call the function to get the coefficients
getOlsCoefs()


Overall rating              0.439050
Height                      0.047466
Weight                     -0.040326
Growth                      0.107944
Total attacking            -0.023048
Crossing                    0.009608
Short passing              -0.027449
Volleys                    -0.001096
Dribbling                   0.061930
Total defending             0.009413
International reputation    0.474637
Name: ols, dtype: float64

In [4]:
# HIDDEN TEST CELL


**Question 2:** Calculate the Variance Inflation Factor (VIF) of each variable in this model. 

In [5]:
def getVIFs():
    # Select all features excluding 'Wage', 'club_name', and 'player_name'
    inputs = list(fifa.drop(columns=["Wage", "club_name", "player_name"]).columns)
    vifs = []

    def VIF(v, data):
        # Drop rows with NaN across both v and data
        X_vif = data.dropna()
        v_aligned = v[X_vif.index].dropna()
        X_vif = X_vif.loc[v_aligned.index]
        
        # Fit linear regression: predict v from all other features
        model = LinearRegression()
        model.fit(X_vif, v_aligned)
        pred = model.predict(X_vif)
        
        # Compute correlation between actual v and predicted v
        rho = np.corrcoef(v_aligned, pred)[0, 1]
        
        # VIF = 1 / (1 - R^2) where R = rho
        return 1 / (1 - rho**2)

    # Calculate VIF for each feature
    for f in inputs:
        other_features = [c for c in inputs if c != f]
        vif_val = VIF(fifa[f], fifa[other_features])
        vifs.append(vif_val)

    # Return as a labeled Pandas Series
    out = pd.Series(
        data=vifs,
        index=inputs,
        name="VIF"
    )
    return out

getVIFs()


Overall rating               2.295064
Height                       2.899486
Weight                       2.683393
Growth                       1.505777
Total attacking             34.459943
Crossing                     6.015321
Short passing                9.648880
Volleys                     11.142960
Dribbling                   10.381930
Total defending              2.531194
International reputation     1.281632
Name: VIF, dtype: float64

In [6]:
# HIDDEN TEST CELL

